## Functional diversification analysis cell type wise

In [1]:
import pandas as pd

#This consolidated file contains the consolidated data from all samples together. Only 'SMC02-N' information is not here in this file as we have 
#discarded that sample before already.
cons_df = pd.read_csv("../../pipeline_testing/pareto_front_consolidated_data.csv", index_col=0)
cons_df = cons_df[cons_df['archetype'] != 'Transitory_cells']

epithelial_cells = cons_df[cons_df['Cell_type']== "Epithelial cells"]
myeloid_cells = cons_df[cons_df['Cell_type']== "Myeloids"]
t_cells = cons_df[cons_df['Cell_type']== "T cells"]
stromal_cells = cons_df[cons_df['Cell_type']== "Stromal cells"]

In [2]:
import os
import pandas as pd
import glob

samples = ["SMC01-N", "SMC03-N", "SMC04-N", "SMC05-N", "SMC06-N", "SMC07-N", "SMC08-N", "SMC09-N", "SMC10-N",
            "SMC07-T", "SMC24-T", "SMC01-T", "SMC05-T", "SMC09-T", "SMC15-T", "SMC18-T", "SMC02-T", "SMC03-T",
            "SMC04-T", "SMC06-T", "SMC08-T", "SMC10-T", "SMC11-T", "SMC14-T", "SMC16-T", "SMC17-T", "SMC19-T",
            "SMC20-T", "SMC22-T", "SMC23-T", "SMC21-T", "SMC25-T"]

path = '../../pipeline_testing/mellon_analysis/'

def read_csvs_by_folder(parent_directory):
    folder_data_map = {}
    try:
        subfolders = [f for f in samples
                      if os.path.isdir(os.path.join(parent_directory, f))]
    except FileNotFoundError:
        print("The specified parent directory was not found.")
        return {}

    for folder in subfolders:
        files = glob.glob(os.path.join("../../pipeline_testing/mellon_analysis/", folder) + "/*_specialists_and_generalists.csv")
        files.sort()
        folder_data_map[folder] = []
        for i,file in enumerate(files):
            file_names = [os.path.basename(file) for file in files]
            try:
                df = pd.read_csv(file,index_col=0)
                df_epi = df.loc[df.cells.isin(epithelial_cells['index']),:]
                df_myo = df.loc[df.cells.isin(myeloid_cells['index']),:]
                df_t = df.loc[df.cells.isin(t_cells['index']),:]
                df_stromal = df.loc[df.cells.isin(stromal_cells['index']),:]

                df_concat = pd.concat([df_epi.cells.reset_index(drop=True), df_myo.cells.reset_index(drop=True), df_t.cells.reset_index(drop=True), df_stromal.cells.reset_index(drop=True)],
                                       axis=1)
                df_concat.columns = ['Epithelial','Myeloid', 'T', 'Stromal']
                
                export_path = os.path.join("../../pipeline_testing/mellon_celltype_specialist_generalist/",folder)
                print(export_path)
                os.makedirs(export_path, exist_ok=True)

                df_concat.to_csv(os.path.join(export_path,file_names[i]))
                
                folder_data_map[folder].append(df)
            except Exception as e:
                print(f"Could not read file {file}: {e}")

    return folder_data_map

parent_dir = path
sg_cells = read_csvs_by_folder(parent_dir)

../../pipeline_testing/mellon_celltype_specialist_generalist/SMC01-N
../../pipeline_testing/mellon_celltype_specialist_generalist/SMC01-N
../../pipeline_testing/mellon_celltype_specialist_generalist/SMC01-N
../../pipeline_testing/mellon_celltype_specialist_generalist/SMC01-N
../../pipeline_testing/mellon_celltype_specialist_generalist/SMC01-N
../../pipeline_testing/mellon_celltype_specialist_generalist/SMC01-N
../../pipeline_testing/mellon_celltype_specialist_generalist/SMC01-N
../../pipeline_testing/mellon_celltype_specialist_generalist/SMC01-N
../../pipeline_testing/mellon_celltype_specialist_generalist/SMC01-N
../../pipeline_testing/mellon_celltype_specialist_generalist/SMC01-N
../../pipeline_testing/mellon_celltype_specialist_generalist/SMC01-N
../../pipeline_testing/mellon_celltype_specialist_generalist/SMC01-N
../../pipeline_testing/mellon_celltype_specialist_generalist/SMC03-N
../../pipeline_testing/mellon_celltype_specialist_generalist/SMC03-N
../../pipeline_testing/mellon_cell

## Archetype purity check by cell types across different samples

In [1]:
library(dplyr)
cons_df <- read.csv("../../pipeline_testing/pareto_front_consolidated_data.csv", row.names=1)
cons_df_filt <- cons_df[cons_df$Sample != 'SMC02-N',] 
cons_df_filt <- cons_df[cons_df$archetype != "Transitory_cells",]

desired_order <- c("SMC01-N", "SMC03-N", "SMC04-N", "SMC05-N", "SMC06-N", "SMC07-N", "SMC08-N", "SMC09-N", "SMC10-N",
                    "SMC07-T", "SMC24-T", "SMC01-T", "SMC05-T", "SMC09-T", "SMC15-T", "SMC18-T", "SMC02-T", "SMC03-T",
                    "SMC04-T", "SMC06-T", "SMC08-T", "SMC10-T", "SMC11-T", "SMC14-T", "SMC16-T", "SMC17-T", "SMC19-T",
                    "SMC20-T", "SMC22-T", "SMC23-T", "SMC21-T", "SMC25-T")

rearranged_df <- cons_df_filt %>%
                  mutate(Sample = factor(Sample, levels = desired_order)) %>%
                  arrange(Sample)

cell_counts_df <- rearranged_df %>% 
                        count(Sample, Cell_type, name = "cell_count")

cell_counts_df_epithelial <- cell_counts_df[cell_counts_df$Cell_type=='Epithelial cells',]
cell_counts_df_myeloid <- cell_counts_df[cell_counts_df$Cell_type=='Myeloids',]
cell_counts_df_t <- cell_counts_df[cell_counts_df$Cell_type=='T cells',]
cell_counts_df_stromal <- cell_counts_df[cell_counts_df$Cell_type=='Stromal cells',]

# cell_counts_df_epithelial_filt <- cell_counts_df_epithelial[cell_counts_df_epithelial$cell_count > 5,]
# cell_counts_df_myeloid_filt <- cell_counts_df_myeloid[cell_counts_df_myeloid$cell_count > 5,]
# cell_counts_df_t_filt <- cell_counts_df_t[cell_counts_df_t$cell_count > 5,]
# cell_counts_df_stromal_filt <- cell_counts_df_stromal[cell_counts_df_epithelial$cell_count > 5,]


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
path_save <- "../../pipeline_testing/some_extra_plots/"
dir.create(path_save, recursive = TRUE)
setwd(path_save)
library(stringr)
library(tools)
library(glue)
library(ggplot2)
library(tidyr)

sample <- as.character(cell_counts_df_epithelial$Sample)

sample_epithelial <- list()
for (j in 1:length(sample)){
	sg_folder <- list.files(path= paste("../../pipeline_testing/mellon_celltype_specialist_generalist/",sample[j],sep="/"))
	sg_files <- sg_folder[str_detect(sg_folder, ".csv$")]
	sg_epithelial <- list()
    counter <- 1  
    
    for (i in 1:length(sg_files)){
        sg_path <- file.path("../../pipeline_testing/mellon_celltype_specialist_generalist", sample[j], sg_files[[i]])
        df <- read.csv(sg_path, row.names=1)

        # This part will run cell type wise. Hence in line 24-27 please mention the  cell type name carefully.
        # Here we have chosen 'Epithelial' cell type. Make sure the 'Epithelial' column actually exists in mellon_celltype_specialist_generalist file
        if ("Epithelial" %in% colnames(df)) {
            is_valid <- !is.na(df$Epithelial) & df$Epithelial != ""
            if (sum(is_valid) >= 1) { 
                clean_data <- df$Epithelial[is_valid]
                # Store the cleaned data in the list
                sg_epithelial[[counter]] <- clean_data  
                
                # Get filename without extension
                big_word <- file_path_sans_ext(sg_files[i])
                
                # --- NEW LOGIC START ---
                if (startsWith(big_word, "Archetype")) {
                    # 1. Take everything before the string 'specialists'
                    # This removes "_specialists" and everything after it
                    extracted_name <- sub("_specialists.*", "", big_word)
                    
                } else if (startsWith(big_word, "generalists")) {
                    # 2. Directly take the word 'generalists'
                    extracted_name <- "generalists_cell"
                    
                } else {
                    # 3. Fallback to your original logic if it's neither
                    extracted_name <- word(big_word, start = 1, end = 2, sep = "_")
                }
                names(sg_epithelial[[counter]]) <- extracted_name
                
                counter <- counter + 1
            }
        }     
    }
    sample_epithelial[[j]] <- sg_epithelial
}

Warning message in dir.create(path_save, recursive = TRUE):
“'../../pipeline_testing/some_extra_plots' already exists”


In [4]:
library(ggplot2)
library(dplyr)
library(viridis)

if(is.null(names(sample_epithelial))) {
  names(sample_epithelial) <- sample
}

# Create an empty table to store our final results
plot_data <- data.frame()

for(i in 1:length(sample_epithelial)) {
  
    sample_name <- names(sample_epithelial)[[i]]  
    current_sample <- sample_epithelial[[i]]  
    
    sublist_names<- list()
    for (j in 1:length(current_sample)){
        sublist_names[[j]] <- names(current_sample[[j]][1])
    }
    
    cell_counts <- sapply(current_sample, length)
    
    total_cells <- sum(cell_counts)
    percentage <- (cell_counts / total_cells) * 100
    
    temp_table <- as.data.frame(cbind(sample_name, sublist_names, percentage))
      
  # Combine it with our main empty table
  plot_data <- rbind(plot_data, temp_table)
}

# Remove row names to keep the table clean
# rownames(plot_data) <- NULL

plot_data$sample_name <- sapply(plot_data$sample_name, paste, collapse = ", ")
plot_data$sublist_names <- sapply(plot_data$sublist_names, paste, collapse = ", ")
plot_data$percentage <- as.numeric(plot_data$percentage)

plot_data_order <- plot_data %>%
                      mutate(sample_name = factor(sample_name, levels = sample)) %>%
                      arrange(sample_name)

sg_files_filt <- list()
for (i in 1:length(sample)){
	sg_folder <- list.files(path= paste("../../pipeline_testing/mellon_celltype_specialist_generalist",sample[i],sep="/"))
	sg_files <- sg_folder[str_detect(sg_folder, ".csv$")]
    temp_df = plot_data_order[plot_data_order$sample_name==sample[i],]
    target_name <- temp_df$sublist_names
    combined_pattern <- paste(target_name, collapse = "|")
    clean_pattern <- gsub(" ", "", combined_pattern)   # Removes spaces
    clean_pattern <- gsub("\\|$", "", clean_pattern)
    matched_sg_files <- grep(clean_pattern, sg_files, value = TRUE, ignore.case = TRUE)
    sg_files_filt[[i]] <- matched_sg_files
}

func_all_sample <- list()
for (j in 1:length(sg_files_filt)){
    func_sample<- list()
    for (k in 1:length(sg_files_filt[[j]])){
        data <- read.csv(paste(paste("../../pipeline_testing/functional_characterization_gavish/", sample[j],sep="/"),sg_files_filt[[j]][k],sep="/"))
        data <- data[data$Direction == "Enriched in Archetype",]
        if (nrow(data) == 0) {
            func_sample[[k]] <- NA
            next 
        }
        
        # 3. Add na.rm = TRUE to prevent errors if missing values exist
        func <- data %>%
              filter(Q_Value == min(Q_Value, na.rm = TRUE)) %>%
              filter(Log2FC == max(Log2FC, na.rm = TRUE)) %>%
              slice(1) 
              
        func_sample[[k]] <- func$Feature
    }
    func_all_sample[[j]] <- func_sample
}      

flattened_vector <- unlist(func_all_sample, use.names = FALSE)
plot_data_order$pathway <- flattened_vector
write.csv(plot_data_order, "../../pipeline_testing/some_extra_plots/epithelial_archetype_functional_identity_gavish.csv")

Loading required package: viridisLite



In [6]:
## This is the output that we obtained from all different celltypes
plot_data_order_epi <- read.csv("../../pipeline_testing/some_extra_plots/epithelial_archetype_functional_identity_gavish.csv", row.names=1)
plot_data_order_myo <- read.csv("../../pipeline_testing/some_extra_plots/myeloid_archetype_functional_identity_gavish.csv", row.names=1)
plot_data_order_t <- read.csv("../../pipeline_testing/some_extra_plots/T_archetype_functional_identity_gavish.csv", row.names=1)
plot_data_order_stromal <- read.csv("../../pipeline_testing/some_extra_plots/stromal_archetype_functional_identity_gavish.csv", row.names=1)
all_pathway <- c(plot_data_order_epi$pathway, plot_data_order_myo$pathway, plot_data_order_t$pathway, plot_data_order_stromal$pathway)
all_pathway <- unique(all_pathway)

In [7]:
all_pathway

[1] "GAVISH_3CA_MALIGNANT_METAPROGRAM_30_PDAC_CLASSICAL"        
 [2] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_COLON_RELATED"           
 [3] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_EPISEN"                  
 [4] "GAVISH_3CA_METAPROGRAM_FIBROBLASTS_INTERFERON"             
 [5] "GAVISH_3CA_MALIGNANT_METAPROGRAM_38_GLUTATHIONE"           
 [6] "GAVISH_3CA_METAPROGRAM_FIBROBLASTS_METAL_RESPONSE"         
 [7] "GAVISH_3CA_METAPROGRAM_CD8_T_CELLS_DYSFUNCTION"            
 [8] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_PDAC_RELATED_3"          
 [9] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_MYC"                     
[10] "GAVISH_3CA_METAPROGRAM_CD8_T_CELLS_CYTOTOXIC"              
[11] "GAVISH_3CA_METAPROGRAM_FIBROBLASTS_MHC_II"                 
[12] NA                                                          
[13] "GAVISH_3CA_METAPROGRAM_CD8_T_CELLS_UNASSIGNED_1"           
[14] "GAVISH_3CA_MALIGNANT_METAPROGRAM_23_SECRETED_2"            
[15] "GAVISH_3CA_METAPROGRAM_CD8_T_CELLS_NAIVE_2"                
[16] "GAVISH_3CA_METAPROGRAM_FIBROBLASTS_CAF_5"                  
[17] "GAVISH_3CA_METAPROGRAM_FIBROBLASTS_CAF_1"                  
[18] "GAVISH_3CA_METAPROGRAM_FIBROBLASTS_CAF_2"                  
[19] "GAVISH_3CA_METAPROGRAM_CD4_T_CELLS_NAIVE_1"                
[20] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_RESPIRATION"             
[21] "GAVISH_3CA_METAPROGRAM_FIBROBLASTS_PERICYTE_LIKE"          
[22] "GAVISH_3CA_MALIGNANT_METAPROGRAM_8_PROTEASOMAL_DEGRADATION"
[23] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_ANDROGEN_PROSTATE"       
[24] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_METABOLISM_KIDNEY_2"     
[25] "GAVISH_3CA_MALIGNANT_METAPROGRAM_11_TRANSLATION_INITIATION"
[26] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_EPI_1"                   
[27] "GAVISH_3CA_METAPROGRAM_CD8_T_CELLS_MEMORY_NAIVE_1"         
[28] "GAVISH_3CA_MALIGNANT_METAPROGRAM_12_EMT_1"                 
[29] "GAVISH_3CA_METAPROGRAM_CD4_T_CELLS_T_REG"                  
[30] "GAVISH_3CA_METAPROGRAM_CD4_T_CELLS_GLYCOLYSIS_MYC"         
[31] "GAVISH_3CA_METAPROGRAM_FIBROBLASTS_CAF_6"                  
[32] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_INTERFERON_MHC"          
[33] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_SECRETED"                
[34] "GAVISH_3CA_MALIGNANT_METAPROGRAM_21_RESPIRATION"           
[35] "GAVISH_3CA_METAPROGRAM_MACROPHAGES_MONOCYTE_SECRETED"      
[36] "GAVISH_3CA_METAPROGRAM_CD8_T_CELLS_GLYCOLYSIS_MYC"         
[37] "GAVISH_3CA_METAPROGRAM_ENDOTHELIAL_HEV_1"                  
[38] "GAVISH_3CA_MALIGNANT_METAPROGRAM_19_EPITHELIAL_SENESCENCE" 
[39] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_PDAC_RELATED_2"          
[40] "GAVISH_3CA_MALIGNANT_METAPROGRAM_20_MYC"                   
[41] "GAVISH_3CA_MALIGNANT_METAPROGRAM_40_PDAC_RELATED"          
[42] "GAVISH_3CA_MALIGNANT_METAPROGRAM_3_CELL_CYLCE_HMG_RICH"    
[43] "GAVISH_3CA_METAPROGRAM_CD4_T_CELLS_NAIVE_2"                
[44] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_NEUROENDOCRINE"          
[45] "GAVISH_3CA_METAPROGRAM_MACROPHAGES_MYC_MITOCHONDRIA"       
[46] "GAVISH_3CA_MALIGNANT_METAPROGRAM_26_NPC_GLIOMA"            
[47] "GAVISH_3CA_MALIGNANT_METAPROGRAM_24_CILIA"                 
[48] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_EPI4"                    
[49] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_ALVEOLAR"                
[50] "GAVISH_3CA_METAPROGRAM_EPITHELIAL_PDAC_RELATED_1"          
[51] "GAVISH_3CA_METAPROGRAM_MACROPHAGES_MAC_3"                  
[52] "GAVISH_3CA_MALIGNANT_METAPROGRAM_36_IG"                    
[53] "GAVISH_3CA_METAPROGRAM_MACROPHAGES_LIPID_ASSOCIATED"       
[54] "GAVISH_3CA_METAPROGRAM_FIBROBLASTS_HYPOXIA"                
[55] "GAVISH_3CA_METAPROGRAM_CD4_T_CELLS_CYTOTOXIC"              
[56] "GAVISH_3CA_METAPROGRAM_MACROPHAGES_MES_GLYCOLYSIS"         
[57] "GAVISH_3CA_METAPROGRAM_CD8_T_CELLS_UNASSIGNED_2"           
[58] "GAVISH_3CA_METAPROGRAM_CD4_T_CELLS_DYSFUNCTION"            
[59] "GAVISH_3CA_METAPROGRAM_CD4_T_CELLS_UNASSIGNED"             
[60] "GAVISH_3CA_METAPROGRAM_ENDOTHELIAL_ENDO_3"                 
[61] "GAVISH_3CA_MALIGNANT_METAPROGRAM_15

In [8]:
library(ggplot2)
library(dplyr)
library(viridis)
library(viridisLite)
library(pals)

all_possible_groups <- all_pathway
my_pal <- colorRampPalette(polychrome())(length(all_possible_groups))
names(my_pal) <- all_possible_groups

# plot for 'Epithelial' cell types
pdf("../../pipeline_testing/some_extra_plots/Functional_diversification_gavish_in_epithelial_cells_across_all_samples.pdf",width=25, height=16)
ggplot(plot_data_order, aes(x = sample_name, y = percentage, fill = pathway)) +
  geom_bar(stat = "identity", color = NA, size = 0.3) + 
  labs(
    title = "Functional diversification in Epithelial cells across samples",
    x = "Sample Names",
    y = "Percentage (%)",
    fill = "Functional identity specialist & Generalist"
  ) +
  theme_minimal(base_size = 18) +
  scale_fill_manual(values = my_pal) +
  theme(
    plot.title = element_text(size = 35, face = "bold", hjust = 0.1),
    axis.title = element_text(size = 30, face = "bold"),
    axis.text.x = element_text(color = 'black', angle = 60, hjust = 1, size = 25),
    axis.text.y = element_text(color = 'black', size = 30),
    legend.title = element_text(size = 30, face = "bold"),
    legend.text = element_text(size = 22)
  )+
  guides(fill = guide_legend(ncol = 1))
dev.off()


Attaching package: ‘pals’


The following objects are masked from ‘package:viridis’:

    cividis, inferno, magma, plasma, turbo, viridis


The following objects are masked from ‘package:viridisLite’:

    cividis, inferno, magma, plasma, turbo, viridis


Warning message in geom_bar(stat = "identity", color = NA, size = 0.3):
“Ignoring unknown parameters: `size`”


agg_record_3bcf42ae0a7 
                     2

## Identify cell types distributions in number of archetypes

In [27]:
path_save <- "../../pipeline_testing/some_extra_plots/"
dir.create(path_save, recursive = TRUE)
setwd(path_save)
library(stringr)
library(tools)
library(glue)
library(ggplot2)
library(tidyr)

sample <- as.character(cell_counts_df_stromal$Sample)

sample_epithelial <- list()
for (j in 1:length(sample)){
	sg_folder <- list.files(path= paste("../../pipeline_testing/mellon_celltype_specialist_generalist/",sample[j],sep="/"))
	sg_files <- sg_folder[str_detect(sg_folder, ".csv$")]
	sg_epithelial <- list()
    counter <- 1  # We use this so the final list has NO blank spaces
    
    for (i in 1:length(sg_files)){
        sg_path <- file.path("../../pipeline_testing/mellon_celltype_specialist_generalist", sample[j], sg_files[[i]])
        df <- read.csv(sg_path, row.names=1)

        # STEP 1: Make sure the 'epithelial' column actually exists in this file
        if ("Stromal" %in% colnames(df)) {
            is_valid <- !is.na(df$Stromal) & df$Stromal != ""
            if (sum(is_valid) >= 1) { 
                clean_data <- df$Stromal[is_valid]
                # Store the cleaned data in the list
                sg_epithelial[[counter]] <- clean_data  
                
                # Get filename without extension
                big_word <- file_path_sans_ext(sg_files[i])
                
                # --- NEW LOGIC START ---
                if (startsWith(big_word, "Archetype")) {
                    # 1. Take everything before the string 'specialists'
                    # This removes "_specialists" and everything after it
                    extracted_name <- sub("_specialists.*", "", big_word)
                    
                } else if (startsWith(big_word, "generalists")) {
                    # 2. Directly take the word 'generalists'
                    extracted_name <- "generalists_cell"
                    
                } else {
                    # 3. Fallback to your original logic if it's neither
                    extracted_name <- word(big_word, start = 1, end = 2, sep = "_")
                }
                names(sg_epithelial[[counter]]) <- extracted_name
                
                counter <- counter + 1
            }
        }     
    }
    sample_epithelial[[j]] <- sg_epithelial
}

library(ggplot2)
library(dplyr)
library(viridis)

if(is.null(names(sample_epithelial))) {
  names(sample_epithelial) <- sample
}

# Create an empty table to store our final results
plot_data <- data.frame()

for(i in 1:length(sample_epithelial)) {
  
    sample_name <- names(sample_epithelial)[[i]]  
    current_sample <- sample_epithelial[[i]]  
    
    sublist_names<- list()
    for (j in 1:length(current_sample)){
        sublist_names[[j]] <- names(current_sample[[j]][1])
    }
    
    cell_counts <- sapply(current_sample, length)
    
    total_cells <- sum(cell_counts)
    percentage <- (cell_counts / total_cells) * 100
    
    temp_table <- as.data.frame(cbind(sample_name, sublist_names, percentage))
      
  # Combine it with our main empty table
  plot_data <- rbind(plot_data, temp_table)
}

# Remove row names to keep the table clean
# rownames(plot_data) <- NULL

plot_data$sample_name <- sapply(plot_data$sample_name, paste, collapse = ", ")
plot_data$sublist_names <- sapply(plot_data$sublist_names, paste, collapse = ", ")
plot_data$percentage <- as.numeric(plot_data$percentage)

plot_data_order <- plot_data %>%
                      mutate(sample_name = factor(sample_name, levels = sample)) %>%
                      arrange(sample_name)

sg_files_filt <- list()
for (i in 1:length(sample)){
	sg_folder <- list.files(path= paste("../../pipeline_testing/mellon_celltype_specialist_generalist",sample[i],sep="/"))
	sg_files <- sg_folder[str_detect(sg_folder, ".csv$")]
    temp_df = plot_data_order[plot_data_order$sample_name==sample[i],]
    target_name <- temp_df$sublist_names
    combined_pattern <- paste(target_name, collapse = "|")
    clean_pattern <- gsub(" ", "", combined_pattern)   # Removes spaces
    clean_pattern <- gsub("\\|$", "", clean_pattern)
    matched_sg_files <- grep(clean_pattern, sg_files, value = TRUE, ignore.case = TRUE)
    sg_files_filt[[i]] <- matched_sg_files
}

func_all_sample <- list()
for (j in 1:length(sg_files_filt)){
    func_sample<- list()
    for (k in 1:length(sg_files_filt[[j]])){
        data <- read.csv(paste(paste("../../pipeline_testing/functional_characterization_gavish/", sample[j],sep="/"),sg_files_filt[[j]][k],sep="/"))
        data <- data[data$Direction == "Enriched in Archetype",]
        if (nrow(data) == 0) {
            func_sample[[k]] <- NA
            next 
        }
        
        # 3. Add na.rm = TRUE to prevent errors if missing values exist
        func <- data %>%
              filter(Q_Value == min(Q_Value, na.rm = TRUE)) %>%
              filter(Log2FC == max(Log2FC, na.rm = TRUE)) %>%
              slice(1) 
              
        func_sample[[k]] <- func$Feature
    }
    func_all_sample[[j]] <- func_sample
}      

flattened_vector <- unlist(func_all_sample, use.names = FALSE)
plot_data_order$pathway <- flattened_vector

Warning message in dir.create(path_save, recursive = TRUE):
“'../../pipeline_testing/some_extra_plots' already exists”


In [21]:
#For Epithelial cells. To get this dataframe run the code snippet 9 by chosing this specific cell type. 
#Just change line no 10 and 24-27 in code snippet 9
plot_data_epithelial <- plot_data[plot_data$sublist_name!='generalists cell',] 
plot_data_epithelial <- plot_data_epithelial %>% count(sample_name)
plot_data_epithelial <- plot_data_epithelial %>%
                      arrange(factor(sample_name, levels = sample))

In [26]:
#For T cells. To get this dataframe run the code snippet 9 by chosing this specific cell type
#Just change line no 10 and 24-27 in code snippet 9
plot_data_t <- plot_data[plot_data$sublist_name!='generalists cell',] 
plot_data_t <- plot_data_t %>% count(sample_name)
plot_data_t <- plot_data_t %>%
                      arrange(factor(sample_name, levels = sample))

In [24]:
#For Myeloids. To get this dataframe run the code snippet 9 by chosing this specific cell type
#Just change line no 10 and 24-27 in code snippet 9
plot_data_myeloid <- plot_data[plot_data$sublist_name!='generalists cell',] 
plot_data_myeloid <- plot_data_myeloid %>% count(sample_name)
plot_data_myeloid <- plot_data_myeloid %>%
                      arrange(factor(sample_name, levels = sample))

In [28]:
#For Stromal cells. To get this dataframe run the code snippet 9 by chosing this specific cell type
#Just change line no 10 and 24-27 in code snippet 9
plot_data_stromal <- plot_data[plot_data$sublist_name!='generalists cell',] 
plot_data_stromal <- plot_data_stromal %>% count(sample_name)
plot_data_stromal <- plot_data_stromal %>%
                      arrange(factor(sample_name, levels = sample))

In [29]:
plot_data

sample_name,sublist_names,percentage
<chr>,<chr>,<dbl>
SMC01-N,Archetype_2_0,8.3333333
SMC01-N,Archetype_2_2,0.8333333
SMC01-N,Archetype_2_3,9.7222222
SMC01-N,Archetype_2_4,1.1111111
SMC01-N,Archetype_2_5,9.4444444
SMC01-N,Archetype_2_6,2.7777778
SMC01-N,Archetype_4,12.5000000
SMC01-N,Archetype_5,21.9444444
SMC01-N,generalists_cell,33.3333333


In [30]:
plot_data_list <- list(plot_data_epithelial, plot_data_t, plot_data_myeloid, plot_data_stromal)
merged_df <- Reduce(function(x, y) merge(x, y, by = "sample_name", all = TRUE), plot_data_list)
merged_df[is.na(merged_df)] <- 0
colnames(merged_df) <- c("sample_name", "Epithelial", "T", "Myeloid", "Stromal")
no_archetype_share <- merged_df
rownames(no_archetype_share) <- merged_df$sample_name
no_archetype_share <- subset(no_archetype_share, select = -c(sample_name))
no_archetype_share <- t(no_archetype_share)
no_archetype_share

Warning message in merge.data.frame(x, y, by = "sample_name", all = TRUE):
“column names ‘n.x’, ‘n.y’ are duplicated in the result”


,SMC01-N,SMC01-T,SMC02-T,SMC03-N,SMC03-T,SMC04-N,SMC04-T,SMC05-N,SMC05-T,SMC06-N,⋯,SMC16-T,SMC17-T,SMC18-T,SMC19-T,SMC20-T,SMC21-T,SMC22-T,SMC23-T,SMC24-T,SMC25-T
Epithelial,4,3,2,3,2,3,2,2,3,2,⋯,4,4,4,4,2,5,7,3,2,3
T,3,6,2,4,2,2,2,2,5,4,⋯,2,3,3,3,2,3,2,2,2,4
Myeloid,2,5,3,0,2,2,3,1,3,1,⋯,3,4,2,3,2,6,1,2,3,2
Stromal,9,5,2,2,2,3,1,3,1,6,⋯,0,3,2,2,2,1,5,1,1,2


In [31]:
# Load the package
library(gplots)
library(viridis)

col_annotation <- c(rep("blue", 9), rep("dodgerblue", 2), rep("forestgreen", 5), rep("orangered", 14), rep("maroon", 2))

pdf("../../pipeline_testing/some_extra_plots/No of archetype shared by cell type.pdf", width = 24, height=16)
heatmap.2(
  x = no_archetype_share,
  col = viridis(8),
  Rowv = FALSE,
  Colv = FALSE,
  dendrogram = "none",  
  cexRow = 3,          
  cexCol = 2.7,          
  margins = c(12, 12),   
  srtCol = 45, 
  keysize = 1,
  key = TRUE,
  key.title = "Number of archetype shared",        
  key.xlab = "Value",
  # Your large font sizes for the key
  key.par = list(
    cex = 1.5,        
    cex.axis = 1.5,   
    cex.lab = 1.5,     
    cex.main = 1.5
  ),
  ColSideColors = col_annotation, 
  
  # ====================================================================
  # CRITICAL FIX 1: lhei (Vertical Layout)
  # Changed from c(1.2, 0.15, 4.0) to c(2.5, 0.5, 5.0)
  # - 2.5 gives the top row enough height so the enlarged key doesn't crash
  # - 0.5 gives the middle bar enough room so the color bar doesn't crash
  # - 5.0 holds the main heatmap
  # ====================================================================
  lhei = c(1.5,2.5),   
  
  # ====================================================================
  # CRITICAL FIX 2: lwid (Horizontal Layout)
  # Changed from c(1.5, 12) to c(4.0, 12)
  # - 4.0 stretches the left column to give the key text room to exist
  # - 12 maintains the wide stretch for your 26 columns
  # ====================================================================
  lwid = c(0.5, 1.5),      
  
  # --- CLEANUP ---
  trace = "none",        
  density.info = "none", 
)

par(xpd = TRUE) 
legend("topright",                   
       legend = c("Normal", "Stage 1", "Stage 2", "Stage 3", "Stage 4"), 
       fill = c("Blue", "dodgerblue", "forestgreen", "orangered", "maroon"), 
       border = FALSE,                
       bty = "n",                      
       y.intersp = 0.8,
       cex = 3) 
dev.off()


---------------------
gplots 3.3.0 loaded:
  * Use citation('gplots') for citation info.
  * Homepage: https://talgalili.github.io/gplots/
  * Report issues: https://github.com/talgalili/gplots/issues
  * Ask questions: https://stackoverflow.com/questions/tagged/gplots
  * Suppress this message with: suppressPackageStartupMessages(library(gplots))
---------------------



Attaching package: ‘gplots’


The following object is masked from ‘package:stats’:

    lowess




agg_record_3bcf586d6d72 
                      2